In [2]:
import os
import re
import glob
from IPython.display import display
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
#import japanize_matplotlib
import seaborn as sns 
import unicodedata
import yaml


In [3]:
# configの読み込み
CONFIG_FILE = '../configs/config.yaml'
with open(CONFIG_FILE, encoding="utf-8") as file:
    yml = yaml.safe_load(file)

In [4]:
DIR_INPUT = yml["SETTING"]["DIR_INPUT"]
DIR_INTERIM = yml["SETTING"]["DIR_INTERIM"]
DIR_FEATURE = yml["SETTING"]["DIR_FEATURE"]
FILE_NAME_STATION = yml["SETTING"]["FILE_NAME_STATION"]
FILE_NAME_STATUS = yml["SETTING"]["FILE_NAME_STATUS"]
FILE_NAME_TRIP = yml["SETTING"]["FILE_NAME_TRIP"]
FILE_NAME_WEATHER = yml["SETTING"]["FILE_NAME_WEATHER"]
DIR_FIGURE = yml["SETTING"]["DIR_FIGURE"]

# データ読み込み

In [27]:
dict_dtype_station = {
    "station_id": "int64",
    "lat": "float64",
    "long": "float64",
    "dock_count": "int64",
    "city": "object",
    "installation_date": "object",
}
dict_dtype_status = {
    "id": "int64",
    "year": "int64",
    "month": "int64",
    "day": "int64",
    "hour": "int64",
    "station_id": "int64",
    "bikes_available": "float64",
    "predict": "int64",
}
dict_dtype_trip = {
    "trip_id": "int64",
    "duration": "int64",
    "start_date": "object",
    "start_station_id": "int64",
    "end_date": "object",
    "end_station_id": "int64",
    "bike_id": "int64",
    "subscription_type": "object",
}
dict_dtype_weather = {
    "date": "object",
    "max_temperature": "int64",
    "mean_temperature": "int64",
    "min_temperature": "int64",
    "max_dew_point": "int64",
    "mean_dew_point": "int64",
    "min_dew_point": "int64",
    "max_humidity": "int64",
    "mean_humidity": "int64",
    "min_humidity": "int64",
    "max_sea_level_pressure": "float64",
    "mean_sea_level_pressure": "float64",
    "min_sea_level_pressure": "float64",
    "max_visibility": "int64",
    "mean_visibility": "int64",
    "min_visibility": "int64",
    "max_wind_Speed": "int64",
    "mean_wind_speed": "int64",
    "precipitation": "float64",
    "cloud_cover": "int64",
    "events": "object",
    "wind_dir_degrees": "int64",
}

In [28]:
df_station = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATION), dtype=dict_dtype_station)
df_status = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATUS), dtype=dict_dtype_status)
df_trip = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TRIP), dtype=dict_dtype_trip)
df_weather = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_WEATHER), dtype=dict_dtype_weather)

# 前処理

In [35]:
def convert_datetime(df_station, df_status, df_trip, df_weather):
    df_station["installation_date"] = pd.to_datetime(df_station["installation_date"], format='%m/%d/%Y').dt.date
    df_status["datetime"] = pd.to_datetime(df_status[["year", "month", "day", "hour"]])
    df_trip["start_date"] = pd.to_datetime(df_trip["start_date"], format='%m/%d/%Y %H:%M')
    df_trip["end_date"] = pd.to_datetime(df_trip["end_date"], format='%m/%d/%Y %H:%M')
    df_weather["date"] = pd.to_datetime(df_weather["date"], format='%Y-%m-%d').dt.date

    return df_station, df_status, df_trip, df_weather

In [36]:
df_station, df_status, df_trip, df_weather = convert_datetime(df_station, df_status, df_trip, df_weather)

## データ出力

In [37]:
df_station.to_pickle(os.path.join(DIR_INTERIM, "df_prep_station.pkl"))
df_status.to_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))
df_trip.to_pickle(os.path.join(DIR_INTERIM, "df_prep_trip.pkl"))
df_weather.to_pickle(os.path.join(DIR_INTERIM, "df_prep_weather.pkl"))

# 特徴量作成

In [38]:
from pathlib import Path
from abc import ABCMeta, abstractmethod

In [39]:
from time import time

def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20
        
    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [41]:
class AbstractBaseBlock(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        
    # 内部状態の更新
    def fit(self, df_input: pd.DataFrame, y=None):
        pass
    
    # 変換処理
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError()
    
    # 内部状態の更新と変換処理をまとめて行う
    def fit_transform(self, X: pd.DataFrame, y=None):
        self.fit(X, y)
        return self.transform(X)
    
    # 特徴量生成処理
    def create_feature(self, X, y=None, test=False, limit_date="yyyy-mm-dd") -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, limit_date, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            return pd.read_pickle(file_name)
        
        # 変換処理を実行
        else:
            # trainの場合
            if not test:
                feature = self.fit_transform(X, y)
            # testの場合
            else:
                feature = self.transform(X)
            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

            return feature

In [9]:
# そのままの特徴量を返すBlock

In [ ]:
# 静的特徴量
# 動的特徴量

In [209]:
class AsIsNumetricBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'CityTier', 
            'DurationOfPitch',
            'NumberOfPersonVisiting',
            'NumberOfFollowups', 
            'PreferredPropertyStar', 
            'NumberOfTrips',
            'Passport', 
            'PitchSatisfactionScore',
            'MonthlyIncome', 
            'ProdTaken' # 目的関数
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

class AsIsCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'Age', 
            'TypeofContact',
            'Occupation',
            'Gender',
            'ProductPitched',
            'Designation', 
            'customer_info', 
            'marry', 
            'car', 
            'child',
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

In [73]:
class CountEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_count = None

    def fit(self, df_input, y=None):
        # カラムごとにマッピング表を作成
        self.map_count = {}
        for col in self.use_cols:
            self.map_count[col] = df_input[col].fillna("NA").value_counts()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].fillna("NA").map(self.map_count[col]).astype(int)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('CE_')], axis=1)
        return df_result
        
class TargetEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y):
        # カラムごとにマッピング表を作成
        self.map_target = {}
        for col in self.use_cols:
            self.map_target[col] = df_input.groupby(col)[TARGET_COL].mean()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].map(self.map_target[col]).astype(float)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('TE_')], axis=1)
        return df_result
    
# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result
    
    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result
    
# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result
    
    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result

In [75]:
def run_blocks(df_input, feature_blocks, y=None, test=False):
    df_out = None
    
    print(decorate('start run blocks...'))

    with Timer(prefix='run test={}'.format(test)):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature(df_input, y=y, test=test)
            assert len(df_input) == len(feature), block
            if df_out is None:
                df_out = feature
            else:
                df_out = pd.merge(df_out, feature, on=["id"], how="left")

    return df_out

In [210]:
feature_blocks = [
    *[AsIsCategoryBlock(use_cache=False, save_cache=True, logger=None)],
    *[AsIsNumetricBlock(use_cache=True, save_cache=True, logger=None)],
    *[CountEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[TargetEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[BigCategoryBlock(use_cache=True, save_cache=True, logger=None)],
]

In [211]:
df_preprocessed = pd.read_pickle(os.path.join(DIR_INTERIM, "preprocessed.pkl"))

In [212]:
df_out = run_blocks(df_preprocessed, feature_blocks, y=None, test=False)

★★★★★★★★★★★★★★★★★★★★ start run blocks... ★★★★★★★★★★★★★★★★★★★★
	- <__main__.AsIsCategoryBlock object at 0x7fd57ad0a400> 0.034[s]
	- <__main__.AsIsNumetricBlock object at 0x7fd57ad0a8b0> 0.004[s]
	- <__main__.CountEncodingBlock object at 0x7fd57ad0a8e0> 0.002[s]
	- <__main__.TargetEncodingBlock object at 0x7fd57ad0a940> 0.001[s]
	- <__main__.BigCategoryBlock object at 0x7fd57ad0ac70> 0.001[s]
run test=False 0.069[s]


In [114]:
df_status = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))

In [115]:
df_status_feat = df_status.copy()
df_status_feat["weekdays"] = df_status["datetime"].dt.weekday

In [116]:
#これから細かい前処理をするためにmain_dfを作成
main_df = df_status_feat[["datetime", "year", "month", "day", "hour", "station_id","bikes_available","predict", "weekdays"]]
main_df.head()

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2013-09-01 00:00:00,2013,9,1,0,0,11.0,0,6
1,2013-09-01 01:00:00,2013,9,1,1,0,11.0,0,6
2,2013-09-01 02:00:00,2013,9,1,2,0,11.0,0,6
3,2013-09-01 03:00:00,2013,9,1,3,0,11.0,0,6
4,2013-09-01 04:00:00,2013,9,1,4,0,11.0,0,6


In [117]:
#学習用のデータフレームを作成
train_dataset_df = main_df[main_df["datetime"]<"2014-09-01"]
#評価用のデータフレームを作成(使用するモデルの関係上、前日のデータが必要なため2014-08-31から取得)
evaluation_dataset_df = main_df[main_df["datetime"]>="2014-08-31"]

In [118]:
#各ステーション毎に、欠損値を後の値で埋める
train_dataset_df_new = pd.DataFrame()
for station_id in train_dataset_df["station_id"].unique().tolist():
    temp_df = train_dataset_df[train_dataset_df["station_id"]==station_id]
    temp_df = temp_df.fillna(method="bfill")
    train_dataset_df_new = pd.concat([train_dataset_df_new,temp_df])

print(train_dataset_df_new.isnull().sum())

/var/folders/qk/pwcsrt352q5ds79rrtgfbtrh0000gn/T/ipykernel_37765/843399533.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df = temp_df.fillna(method="bfill")


datetime           0
year               0
month              0
day                0
hour               0
station_id         0
bikes_available    0
predict            0
weekdays           0
dtype: int64


In [119]:
#データセットを時系列に並び替える(後ほど説明)
train_df = train_dataset_df_new.sort_values(["datetime","station_id"],ascending=True).reset_index(drop=True)
evaluation_dataset_df = evaluation_dataset_df.sort_values(["datetime","station_id"],ascending=True).reset_index(drop=True)
#学習用データセット
train_df.head()

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2013-09-01,2013,9,1,0,0,11.0,0,6
1,2013-09-01,2013,9,1,0,1,8.0,0,6
2,2013-09-01,2013,9,1,0,2,5.0,0,6
3,2013-09-01,2013,9,1,0,3,9.0,0,6
4,2013-09-01,2013,9,1,0,4,8.0,0,6


In [25]:
24 * 70

1680

# モデル構築

In [57]:
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import math
from statistics import mean
from sklearn.metrics import mean_squared_error

In [121]:
#predictは特徴量として必要ないため、削除
train_df = train_df.drop("predict",axis=1)
train_df = train_df.iloc[:,6:]
train_df.head()

,bikes_available,weekdays
0,11.0,6
1,8.0,6
2,5.0,6
3,9.0,6
4,8.0,6


In [106]:
#学習用のデータをモデルの学習用とモデルの精度の検証用に分割
#今回は、モデル用学習データ:精度用の検証データ = 8 : 2 に分割
length = len(train_df)
train_size = int(length * 0.8)
test_size = length - train_size
train, test = train_df.iloc[0:train_size,:], train_df.iloc[train_size:length,:]
print(train.shape)
print(test.shape)

(490560, 2)
(122640, 2)


In [107]:
#今回LSTMモデルを使用するため、データを標準化
#特徴量を標準化するための変数
scaler = MinMaxScaler(feature_range=(0, 1))
#標準化された出力をもとにスケールに変換(inverse)するために必要な変数
scaler_for_inverse = MinMaxScaler(feature_range=(0, 1))
train_scale = scaler.fit_transform(train)
test_scale = scaler.transform(test)
bikes_available_scale = scaler_for_inverse.fit_transform(train[["bikes_available"]])
print(train_scale.shape)
print(test_scale.shape)

(490560, 2)
(122640, 2)


In [148]:
def create_dataset(dataset):
    dataX = []
    dataY = np.array([])
    #1680で1つのデータセットであるため、余りの分は使わない
    extra_num = len(dataset) % 70
    max_len = len(dataset)-extra_num
    for i in range(1680,max_len,70):
        xset = []
        for j in range(dataset.shape[1]):
            a = dataset[i-1680:i, j]
            xset.append(a)
        temp_array = np.array(dataset[i:i+70,0])
        dataY = np.concatenate([dataY,temp_array])
        dataX.append(xset)
    dataY = dataY.reshape(-1,70)
    return np.array(dataX), dataY 

In [149]:
trainX, trainY = create_dataset(train_scale)
testX, testY = create_dataset(test_scale)
print(trainX.shape)
print(trainY.shape)

(6984, 2, 1680)
(6984, 70)


In [49]:
# データの形状を確認し、PyTorchのテンソルに変換
trainX_tensor = torch.tensor(trainX, dtype=torch.float32)
trainY_tensor = torch.tensor(trainY, dtype=torch.float32)

# データローダーを作成
train_dataset = TensorDataset(trainX_tensor, trainY_tensor)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# LSTMモデルを定義
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1680, hidden_size=50, batch_first=True)
        self.fc = nn.Linear(50, 70)
    
    def forward(self, x):
        h_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        c_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        out, _ = self.lstm(x, (h_0, c_0))
        out = self.fc(out[:, -1, :])
        return out

# モデルのインスタンスを作成
model_pt = LSTMModel()

# 損失関数とオプティマイザを定義
criterion = nn.MSELoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

# モデルをトレーニング
num_epochs = 20
for epoch in range(num_epochs):
    model_pt.train()
    for i, (inputs, targets) in enumerate(train_loader):
        outputs = model_pt(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/20], Loss: 0.0160
Epoch [2/20], Loss: 0.0098
Epoch [3/20], Loss: 0.0151
Epoch [4/20], Loss: 0.0101
Epoch [5/20], Loss: 0.0131
Epoch [6/20], Loss: 0.0043
Epoch [7/20], Loss: 0.0107
Epoch [8/20], Loss: 0.0067
Epoch [9/20], Loss: 0.0136
Epoch [10/20], Loss: 0.0084
Epoch [11/20], Loss: 0.0109
Epoch [12/20], Loss: 0.0117
Epoch [13/20], Loss: 0.0104
Epoch [14/20], Loss: 0.0081
Epoch [15/20], Loss: 0.0070
Epoch [16/20], Loss: 0.0137
Epoch [17/20], Loss: 0.0125
Epoch [18/20], Loss: 0.0158
Epoch [19/20], Loss: 0.0093
Epoch [20/20], Loss: 0.0046


In [66]:
# 
#学習済みモデルで予測
model = model_pt
model.eval()
trainX_tensor = torch.tensor(trainX, dtype=torch.float32)
testX_tensor = torch.tensor(testX, dtype=torch.float32)
train_predict = model(trainX_tensor)
test_predict = model(testX_tensor)

#スケールをもとに戻す
train_predict = scaler_for_inverse.inverse_transform(train_predict.detach().numpy())
trainY = scaler_for_inverse.inverse_transform(trainY)
test_predict = scaler_for_inverse.inverse_transform(test_predict.detach().numpy())
testY = scaler_for_inverse.inverse_transform(testY)

#各ステーションのスコアの平均値を算出
train_score_list = []
test_score_list = []
for i in range(70):
    trainscore = math.sqrt(mean_squared_error(trainY[:,i], train_predict[:,i]))
    train_score_list.append(trainscore)
    testscore = math.sqrt(mean_squared_error(testY[:,i], test_predict[:,i]))
    test_score_list.append(testscore)
    
print("trainのRMSE平均 : ",mean(train_score_list))
print("testのRMSE平均 : ",mean(test_score_list))

trainのRMSE平均 :  2.5844546330388116
testのRMSE平均 :  3.051291571389852


In [86]:
from datetime import timedelta

In [ ]:
# 2. モデルインターフェース例
class ForecastingModelBase:
    def __init__(self, model):
        self.model = model

    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.array:
        """
        1期先の予測を行う関数。
        """
        return self.model.predict(X)
    
class LSTM(ForecastingModelBase):
    def __init__(self, model):
        super().__init__(model)

    def predict(self, X: pd.DataFrame) -> np.array:
        """
        LSTMモデルを使用して1期先の予測を行う関数。
        """
        return self.model.predict(X)

In [ ]:
# 1. 特徴量生成パイプラインの例
def feature_generation_pipeline(df_station: pd.DataFrame, df_status: pd.DataFrame, df_trip: pd.DataFrame, df_weather: pd.DataFrame, target_date: pd.Timestamp) -> pd.DataFrame:
    """
    各種データから特徴量を生成するパイプライン。
    target_dateのデータまでを使用し、ラグ特徴量などを生成する例。
    """
    df_status["weekdays"] = df_status["datetime"].dt.weekday
    
    return df_station, df_status, df_trip, df_weather

# 2. モデルインターフェース例
class ForecastingModelBase:
    def __init__(self, model):
        self.model = model

    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.array:
        """
        1期先の予測を行う関数。
        """
        return self.model.predict(X)
    
class LSTM(ForecastingModelBase):
    def __init__(self, model):
        super().__init__(model)

    def predict(self, X: pd.DataFrame) -> np.array:
        """
        LSTMモデルを使用して1期先の予測を行う関数。
        """
        return self.model.predict(X)

# 3. 再帰的な予測を行う関数
def recursive_forecasting(df_station, df_status, df_trip, df_weather, target_date, model: ForecastingModel, steps: int = 23) -> pd.DataFrame:
    """
    再帰的に特徴量を生成し、1期先から23期先までの予測を行う関数。
    
    df_station: 駅情報のデータフレーム
    df_status: 駅のバイク利用状況のデータフレーム
    df_trip: トリップデータのデータフレーム
    df_weather: 天気データのデータフレーム
    target_date: 予測対象日（target_dateの0時までのデータが使用可能）
    model: 予測モデル (ForecastingModel型)
    steps: 予測する期数 (デフォルトは23期先)
    
    return: 1期先から23期先までの予測結果を含むDataFrame
    """
    
    # 過去データで初期の特徴量生成
    df_station, df_status, df_trip, df_weather = feature_generation_pipeline(df_station, df_status, df_trip, df_weather, target_date)

    # 予測結果を格納するリスト
    predictions = []

    # 再帰的に1期先ずつ予測を行うループ
    for step in range(1, 24):
        
        # モデルで1期先の予測を行う
        model = LSTM(model_pt)
        next_pred = model.predict(df_station, df_status, df_trip, df_weather)
        
        # 予測結果を次のステップの特徴量生成に使用
        next_status = pd.DataFrame(columns=['station_id', 'bikes_available', 'datetime'])
        next_status['bikes_available'] = next_pred
        next_status['station_id'] = [x for x in range(0,70)]
        next_status['datetime'] = target_date + timedelta(hours=step)

        # 予測結果を保存
        predictions.append(next_status)
        
        # 次の期の特徴量を生成（新しい予測値を加える）
        df_status = pd.concat([df_status, next_status], axis=0)
        df_status.sort_values(['datetime', 'station_id'], inplace=True)
        df_station, df_status, df_trip, df_weather = feature_generation_pipeline(df_station, df_status, df_trip, df_weather, target_date)

    # 予測結果をDataFrameとして返す
    return pd.concat(predictions, axis=0)

In [ ]:
# マルチステップモデル

In [88]:
train_df

,bikes_available,weekdays
0,11.0,6
1,8.0,6
2,5.0,6
3,9.0,6
4,8.0,6
...,...,...
613195,10.0,6
613196,9.0,6
613197,7.0,6
613198,8.0,6


In [89]:
evaluation_dataset_df

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2014-08-31 00:00:00,2014,8,31,0,0,11.0,0,6
1,2014-08-31 00:00:00,2014,8,31,0,1,9.0,0,6
2,2014-08-31 00:00:00,2014,8,31,0,2,4.0,0,6
3,2014-08-31 00:00:00,2014,8,31,0,3,8.0,0,6
4,2014-08-31 00:00:00,2014,8,31,0,4,7.0,0,6
...,...,...,...,...,...,...,...,...,...
614875,2015-08-31 23:00:00,2015,8,31,23,65,13.0,0,0
614876,2015-08-31 23:00:00,2015,8,31,23,66,7.0,0,0
614877,2015-08-31 23:00:00,2015,8,31,23,67,6.0,0,0
614878,2015-08-31 23:00:00,2015,8,31,23,68,5.0,0,0


In [108]:
length = len(train_df)
train_size = int(length * 0.8)
test_size = length - train_size
train, test = train_df.iloc[0:train_size,:], train_df.iloc[train_size:length,:]

# データをスケーリング
scaler = MinMaxScaler(feature_range=(0, 1))
train_df_scale = scaler.fit_transform(train)
test_df_scale = scaler.transform(test)

# 入力データ (X) とターゲットデータ (y) を作成
def create_sequences(data, seq_length=24, pred_length=23):
    X, y = [], []
    for i in range(len(data) - seq_length - pred_length):
        X.append(data[i:i+seq_length])  # 特徴量
        y.append(data[i+seq_length:i+seq_length+pred_length, -1])  # bikes_available の未来の値
    return np.array(X), np.array(y)

seq_length = 24  # 24時間の履歴
pred_length = 23  # 23時間先の予測
train_X, train_y = create_sequences(train_df_scale, seq_length, pred_length)
test_X, test_y = create_sequences(test_df_scale, seq_length, pred_length)

In [109]:
train_X.shape, train_y.shape

((490513, 24, 2), (490513, 23))

In [110]:
test_X.shape, test_y.shape

((122593, 24, 2), (122593, 23))

In [120]:
train_dataset_df[train_dataset_df["station_id"]==0]

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2013-09-01 00:00:00,2013,9,1,0,0,11.0,0,6
1,2013-09-01 01:00:00,2013,9,1,1,0,11.0,0,6
2,2013-09-01 02:00:00,2013,9,1,2,0,11.0,0,6
3,2013-09-01 03:00:00,2013,9,1,3,0,11.0,0,6
4,2013-09-01 04:00:00,2013,9,1,4,0,11.0,0,6
...,...,...,...,...,...,...,...,...,...
8755,2014-08-31 19:00:00,2014,8,31,19,0,14.0,0,6
8756,2014-08-31 20:00:00,2014,8,31,20,0,15.0,0,6
8757,2014-08-31 21:00:00,2014,8,31,21,0,15.0,0,6
8758,2014-08-31 22:00:00,2014,8,31,22,0,15.0,0,6


(122593, 23)

１〜２３期先の予測が必要
- bikes_availableを時系列予測

In [ ]:
# i